In [2]:
import duckdb
import pandas as pd
import os
from datetime import datetime

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [4]:
arquivo = "z0019_2.csv"
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-24 09:07:52.779478
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-24 09:07:52.779478
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-24 09:07:52.779478


In [5]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos (
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS INTEGER,
        LABST INTEGER,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
    )
""")

In [6]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [7]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-12 22:05:44.771767
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-11-12 22:05:44.771767
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-11-12 22:05:44.771767
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-12 22:12:20.075385
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-12 22:12:20.075385
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-12 22:12:20.075385


In [9]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

CatalogException: Catalog Error: Could not rename "bronze_produtos" to "bronze_z0019": another entry with this name already exists!

In [10]:
con.close()